In [ ]:
!pip install opencv-python-headless imutils

import cv2
import numpy as np
import os
from google.colab import drive
import matplotlib.pyplot as plt
drive.mount('/content/drive')

def load_db(folder):
    db = {}
    for f in os.listdir(folder):
        name, _ = os.path.splitext(f)
        img = cv2.imread(os.path.join(folder, f))
        if img is not None:
            db[name] = img
    return db

def find_specific_book(test_image_path, book_db, target_book_name, detector, LOWE_RATIO=0.7, MIN_MATCHES=15):
    """
    Search for a specific book in the image instead of finding the best match among all books
    """
    # Check if the target book exists in database
    if target_book_name not in book_db:
        print(f"Error: Book '{target_book_name}' not found in database")
        print(f"Available books: {list(book_db.keys())}")
        return None, None

    test_img = cv2.imread(test_image_path)
    test_gray = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)
    test_kp, test_desc = detector.detectAndCompute(test_gray, None)

    if test_desc is None:
        print("No features detected in test image")
        return None, None

    print(f"Searching for '{target_book_name}' in the image...")

    # Get the specific book cover
    cover = book_db[target_book_name]
    gray_cover = cv2.cvtColor(cover, cv2.COLOR_BGR2GRAY)
    kp_cover, des_cover = detector.detectAndCompute(gray_cover, None)

    if des_cover is None:
        print(f"No features detected in '{target_book_name}' cover")
        return None, None

    # FLANN matcher
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)

    try:
        matches = flann.knnMatch(des_cover, test_desc, k=2)

        # Lowe's ratio test
        good_matches = []
        for match_pair in matches:
            if len(match_pair) == 2:  # Make sure we have both matches
                m, n = match_pair
                if m.distance < LOWE_RATIO * n.distance:
                    good_matches.append(m)

        print(f"Found {len(good_matches)} good matches")

        if len(good_matches) >= MIN_MATCHES:
            src_pts = np.float32([kp_cover[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
            dst_pts = np.float32([test_kp[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

            H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

            if H is not None:
                inliers = np.sum(mask)
                h, w = cover.shape[:2]
                corners = np.float32([[0,0],[w,0],[w,h],[0,h]]).reshape(-1,1,2)
                transformed_corners = cv2.perspectiveTransform(corners, H)

                # Geometric validation
                area = cv2.contourArea(transformed_corners)
                total_image_area = test_img.shape[0] * test_img.shape[1]

                print(f"Book area: {area}, Image area: {total_image_area}, Ratio: {area/total_image_area:.4f}")
                print(f"Inliers: {inliers}/{len(good_matches)}")

                if area > 0.02 * total_image_area:  # At least 2% of the image
                    book_center = np.mean(transformed_corners.reshape(-1, 2), axis=0)
                    image_center = np.array([test_img.shape[1] / 2, test_img.shape[0] / 2])
                    center_distance = np.linalg.norm(book_center - image_center)
                    center_score = 1 - (center_distance / max(test_img.shape[:2]))

                    match_quality = inliers / len(good_matches)
                    area_ratio = area / total_image_area
                    final_score = match_quality * area_ratio * center_score

                    print(f"Match quality: {match_quality:.3f}, Area ratio: {area_ratio:.4f}, Center score: {center_score:.3f}")
                    print(f"Final score: {final_score:.3f}")

                    match_data = {
                        'cover': cover,
                        'homography': H,
                        'corners': transformed_corners,
                        'dst_corners': transformed_corners,
                        'matches': len(good_matches),
                        'inliers': inliers,
                        'score': final_score
                    }

                    return target_book_name, match_data
                else:
                    print(f"Book area too small: {area/total_image_area:.4f} of image")
            else:
                print("Could not find valid homography")
        else:
            print(f"Not enough matches: {len(good_matches)} < {MIN_MATCHES}")

    except Exception as e:
        print(f"Error during matching: {str(e)}")

    return None, None

def get_template_matched_cover(original_cover, input_book_region):
    """Use template matching to get the best aligned reference cover"""
    template = cv2.cvtColor(original_cover, cv2.COLOR_BGR2GRAY)
    scene = cv2.cvtColor(input_book_region, cv2.COLOR_BGR2GRAY)

    scales = np.linspace(0.7, 1.3, 10)
    best_match = None
    best_val = -1
    best_scale = 1.0

    h, w = template.shape

    for scale in scales:
        new_w, new_h = int(w * scale), int(h * scale)
        if new_w <= 0 or new_h <= 0:
            continue
        resized_template = cv2.resize(template, (new_w, new_h))

        if resized_template.shape[0] > scene.shape[0] or resized_template.shape[1] > scene.shape[1]:
            continue

        result = cv2.matchTemplate(scene, resized_template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(result)

        if max_val > best_val:
            best_val = max_val
            best_scale = scale
            best_match = (max_loc, resized_template.shape)

    if best_match is None:
        return original_cover, 1.0, 0.0

    return cv2.resize(original_cover, (int(w * best_scale), int(h * best_scale))), best_scale, best_val


def detect_occlusions(input_image, reference_cover, homography, dst_corners, mask_cover, threshold):
    print("Using histogram equalization method for occlusion detection")

    h, w = reference_cover.shape[:2]
    warped_reference = cv2.warpPerspective(reference_cover, homography, (input_image.shape[1], input_image.shape[0]))

    input_book_region = cv2.bitwise_and(input_image, input_image, mask=mask_cover)
    reference_book_region = cv2.bitwise_and(warped_reference, warped_reference, mask=mask_cover)

    input_gray_eq = cv2.cvtColor(input_book_region, cv2.COLOR_BGR2GRAY)
    reference_gray_eq = cv2.cvtColor(reference_book_region, cv2.COLOR_BGR2GRAY)

    input_gray_eq = cv2.bitwise_and(input_gray_eq, input_gray_eq, mask=mask_cover)
    reference_gray_eq = cv2.bitwise_and(reference_gray_eq, reference_gray_eq, mask=mask_cover)

    equalized_diff = cv2.absdiff(input_gray_eq, reference_gray_eq)
    _, occlusion_mask = cv2.threshold(equalized_diff, threshold, 255, cv2.THRESH_BINARY)
    occlusion_mask = cv2.bitwise_and(occlusion_mask, mask_cover)

    kernel_small = np.ones((3,3), np.uint8)
    occlusion_mask = cv2.morphologyEx(occlusion_mask, cv2.MORPH_OPEN, kernel_small)
    occlusion_mask = cv2.morphologyEx(occlusion_mask, cv2.MORPH_CLOSE, kernel_small)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(occlusion_mask)
    book_area = np.sum(mask_cover > 0)
    min_component_area = book_area * 0.01
    max_component_area = book_area * 0.5

    cleaned_mask = np.zeros_like(occlusion_mask)
    valid_components = 0
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if min_component_area <= area <= max_component_area:
            cleaned_mask[labels == i] = 255
            valid_components += 1

    print(f"Found {valid_components} valid occlusion components")
    occlusion_area = np.sum(cleaned_mask > 0)
    occlusion_ratio = occlusion_area / book_area if book_area > 0 else 0
    print(f"Occlusion ratio: {occlusion_ratio:.3f}")
    print(f"Dynamic threshold used: {threshold}")

    if occlusion_ratio > 0.3:
        print("High occlusion ratio - applying stricter threshold")
        strict_threshold = threshold + 20
        _, strict_mask = cv2.threshold(equalized_diff, strict_threshold, 255, cv2.THRESH_BINARY)
        strict_mask = cv2.bitwise_and(strict_mask, mask_cover)
        strict_mask = cv2.morphologyEx(strict_mask, cv2.MORPH_OPEN, kernel_small)

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(strict_mask)
        cleaned_mask = np.zeros_like(strict_mask)
        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            if min_component_area <= area <= max_component_area:
                cleaned_mask[labels == i] = 255

        occlusion_area = np.sum(cleaned_mask > 0)
        occlusion_ratio = occlusion_area / book_area if book_area > 0 else 0
        print(f"After strict filtering - Occlusion ratio: {occlusion_ratio:.3f}")

    kernel_large = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    cleaned_mask = cv2.morphologyEx(cleaned_mask, cv2.MORPH_CLOSE, kernel_large, iterations=2)
    cleaned_mask = cv2.morphologyEx(cleaned_mask, cv2.MORPH_OPEN, kernel_large, iterations=1)
    cleaned_mask = cv2.GaussianBlur(cleaned_mask, (3,3), 0)
    _, cleaned_mask = cv2.threshold(cleaned_mask, 127, 255, cv2.THRESH_BINARY)

    return cleaned_mask, warped_reference, input_gray_eq, reference_gray_eq, equalized_diff, occlusion_ratio, threshold


def replace_cover_enhanced(title, match_data, input_image, movie_db, threshold, preserve_objects=True):
    if not match_data:
        return None

    cover = match_data['cover']
    H = match_data['homography']
    dst_corners = match_data['dst_corners']
    h, w = cover.shape[:2]

    debug = input_image.copy()
    cv2.polylines(debug, [np.int32(dst_corners)], True, (0,255,0), 3)
    plt.figure(figsize=(8,6))
    plt.title(f"Detected '{title}' - Matches: {match_data['matches']}, Inliers: {match_data['inliers']}")
    plt.axis('off')
    plt.imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
    plt.show()

    if title not in movie_db:
        print(f"  >> No movie poster for '{title}'")
        return None

    poster = movie_db[title]
    poster_resized = cv2.resize(poster, (w, h))
    warped_poster = cv2.warpPerspective(poster_resized, H, (input_image.shape[1], input_image.shape[0]))

    mask_cover = np.zeros(input_image.shape[:2], dtype=np.uint8)
    cv2.fillConvexPoly(mask_cover, np.int32(dst_corners), 255)

    if preserve_objects:
        occlusion_mask, warped_reference, input_eq, reference_eq, diff_image, occlusion_ratio, threshold_used = \
            detect_occlusions(input_image, cover, H, dst_corners, mask_cover, threshold)

        if occlusion_ratio < 0.005 or occlusion_ratio > 0.4:
            print(f"Occlusion ratio {occlusion_ratio:.3f} is outside valid range - skipping object preservation")
            inv_mask = cv2.bitwise_not(mask_cover)
            background = cv2.bitwise_and(input_image, input_image, mask=inv_mask)
            foreground_poster = cv2.bitwise_and(warped_poster, warped_poster, mask=mask_cover)
            return cv2.add(background, foreground_poster)

        inv_mask = cv2.bitwise_not(mask_cover)
        background = cv2.bitwise_and(input_image, input_image, mask=inv_mask)
        foreground_poster = cv2.bitwise_and(warped_poster, warped_poster, mask=mask_cover)
        base_result = cv2.add(background, foreground_poster)

        smooth_occlusion_mask = cv2.GaussianBlur(occlusion_mask, (3,3), 0)
        occlusion_mask_norm = smooth_occlusion_mask.astype(np.float32) / 255.0
        occlusion_mask_3ch = np.stack([occlusion_mask_norm]*3, axis=-1)

        final_result = base_result.astype(np.float32)
        original_with_occlusions = input_image.astype(np.float32)

        final_result = final_result * (1 - occlusion_mask_3ch) + original_with_occlusions * occlusion_mask_3ch
        return final_result.astype(np.uint8)
    else:
        inv_mask = cv2.bitwise_not(mask_cover)
        background = cv2.bitwise_and(input_image, input_image, mask=inv_mask)
        foreground = cv2.bitwise_and(warped_poster, warped_poster, mask=mask_cover)
        return cv2.add(background, foreground)


def process_specific_book(input_image_path, book_db, movie_db, target_book_name, detector, threshold):
    """
    Process image to find and replace a specific book cover
    """
    best_title, best_data = find_specific_book(input_image_path, book_db, target_book_name, detector)

    if best_title is None:
        print(f"Book '{target_book_name}' not detected in the image")
        return None

    input_image = cv2.imread(input_image_path)
    return replace_cover_enhanced(best_title, best_data, input_image, movie_db, threshold, preserve_objects=True)

detector = cv2.SIFT_create()
book_db = load_db('/content/drive/MyDrive/cover_db/covers')
movie_db = load_db('/content/drive/MyDrive/cover_db/posters')

def plot_final_image(result):
    if result is not None:
        plt.figure(figsize=(12,8))
        plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        plt.title('Final Result')
        plt.axis('off')
        plt.show()
    else:
        print("No result to display")



In [ ]:
target_book = "The_Lord_Of_The_Rings_Fellowship"
result = process_specific_book('/content/drive/MyDrive/cover_db/tests/Fellowship.jpeg',
                              book_db, movie_db, target_book, detector, 100)
plot_final_image(result)

In [ ]:
target_book = "The_Lord_Of_The_Rings_Fellowship"
result = process_specific_book('/content/drive/MyDrive/cover_db/tests/Fellowship_2.jpg',
                              book_db, movie_db, target_book, detector, 100)
plot_final_image(result)

In [ ]:
target_book = "The_Hobbit"
result = process_specific_book('/content/drive/MyDrive/cover_db/tests/Hobbit.jpg',
                              book_db, movie_db, target_book, detector, 20)
plot_final_image(result)

In [ ]:
target_book = "The_Hobbit"
result = process_specific_book('/content/drive/MyDrive/cover_db/tests/Hobbit_2.jpg',
                              book_db, movie_db, target_book, detector, 20)
plot_final_image(result)

In [ ]:
target_book = "The_Hobbit"
result = process_specific_book('/content/drive/MyDrive/cover_db/tests/Hobbit_3.jpg',
                              book_db, movie_db, target_book, detector, 45)
plot_final_image(result)

In [ ]:
target_book = "The_Lord_Of_The_Rings_Return"
result = process_specific_book('/content/drive/MyDrive/cover_db/tests/Return.jpg',
                              book_db, movie_db, target_book, detector, 75)
plot_final_image(result)

In [ ]:
target_book = "The_Lord_Of_The_Rings_Towers"
result = process_specific_book('/content/drive/MyDrive/cover_db/tests/Tower.jpg',
                              book_db, movie_db, target_book, detector, 50)
plot_final_image(result)